# Day 29 — Practice Exam Questions

60 practice questions covering all exam domains. For each question:
1. Try to answer without looking at the answer
2. Run the code cell to verify
3. Read the explanation

**Domain weights:** DataFrame API 30% · Architecture 20% · Spark SQL 20% · Optimisation 10% · Streaming 10% · Spark Connect 5% · Pandas API 5%

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import tempfile, os

---
## Section A: DataFrame API (18 questions)

**Q1.** What does `df.count()` return and why does it trigger a Spark job?

In [ ]:
# ANSWER: count() is an ACTION — it forces evaluation of the full DAG.
# Transformations are lazy; only actions trigger execution.
df = spark.range(100)
print(df.count())  # triggers a job

**Q2.** What is the output of `explode` on an empty array vs `explode_outer`?

In [ ]:
arr_df = spark.createDataFrame([
    (1, ["a", "b"]),
    (2, []),
    (3, None),
], ["id", "items"])

# explode: empty array and null → row DROPPED
print("=== explode ===")
arr_df.select("id", explode(col("items")).alias("item")).show()

# explode_outer: keeps the row with null item
print("=== explode_outer ===")
arr_df.select("id", explode_outer(col("items")).alias("item")).show()
# ANSWER: explode drops rows; explode_outer keeps them as null

**Q3.** What is the difference between `rank()` and `dense_rank()` on tied values?

In [ ]:
df = spark.createDataFrame([(1,100),(2,100),(3,90),(4,80)], ["id","score"])
w = Window.orderBy(col("score").desc())
df.withColumn("rank", rank().over(w)) \
  .withColumn("dense_rank", dense_rank().over(w)) \
  .show()
# ANSWER: rank=1,1,3,4 (gap after tie); dense_rank=1,1,2,3 (no gap)

**Q4.** How do you select the top-1 row per group using window functions?

In [ ]:
data = spark.createDataFrame([
    ("Alice","Eng",95000),("Carol","Eng",110000),
    ("Bob","Mkt",72000),("Eve","Mkt",81000),
], ["name","dept","salary"])

w = Window.partitionBy("dept").orderBy(col("salary").desc())
data.withColumn("rn", row_number().over(w)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .show()
# ANSWER: use row_number() + filter(rn==1)

**Q5.** What does `left_semi` join return vs `inner` join?

In [ ]:
emp = spark.createDataFrame([(1,"Alice",10),(2,"Bob",99)],["id","name","dept_id"])
dept = spark.createDataFrame([(10,"Eng")],["dept_id","dept_name"])

print("=== inner ===")     # both sides columns
emp.join(dept, on="dept_id", how="inner").show()
print("=== left_semi ===") # LEFT cols only, only matched rows
emp.join(dept, on="dept_id", how="left_semi").show()
# ANSWER: semi returns only left-side columns; only rows that HAVE a match

**Q6.** How do null keys behave in joins?

In [ ]:
df1 = spark.createDataFrame([(None, "X"), (1, "Y")], ["k", "v1"])
df2 = spark.createDataFrame([(None, "A"), (1, "B")], ["k", "v2"])
df1.join(df2, on="k", how="inner").show()
# ANSWER: null keys NEVER match — the null row is dropped in inner join

**Q7.** What is the return type of `collect_list` vs `collect_set`?

In [ ]:
df = spark.createDataFrame([(1,"a"),(1,"a"),(1,"b"),(2,"c")],["grp","val"])
df.groupBy("grp").agg(
    collect_list("val").alias("list"),  # preserves duplicates
    collect_set("val").alias("set"),    # deduplicates
).show()
# ANSWER: both return ArrayType; collect_list keeps dups, collect_set deduplicates

**Q8.** How do you calculate a 3-row rolling average?

In [ ]:
df = spark.createDataFrame([(1,100),(2,200),(3,150),(4,300),(5,250)],["day","sales"])
w = Window.orderBy("day").rowsBetween(-1, 1)  # current + 1 before + 1 after
df.withColumn("rolling_avg", round(avg("sales").over(w), 2)).show()

**Q9.** What UDF type should you use for performance-critical transformations?

In [ ]:
import pandas as pd

# ANSWER: pandas_udf (vectorized via Arrow) — NOT row-at-a-time udf
@pandas_udf(DoubleType())
def normalize(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

df.withColumn("norm", normalize(col("sales"))).show()

**Q10.** What are the 3 output modes in Structured Streaming?

In [ ]:
# ANSWER:
# append  — only new rows; default; works without aggregation
# complete — entire result table every trigger; requires aggregation
# update  — only changed rows; most memory-efficient with aggregation
print("append | complete | update")

---
## Section B: Architecture (12 questions)

**Q11.** What is the difference between a narrow and wide transformation? Give one example of each.

In [ ]:
# ANSWER:
# Narrow: each output partition depends on ONE input partition — no shuffle
#   examples: filter, select, withColumn, map
# Wide: output partition depends on MULTIPLE input partitions — requires shuffle
#   examples: groupBy, join, distinct, repartition, orderBy
print("Narrow: filter, select | Wide: groupBy, join")

**Q12.** What is a Stage in Spark execution?

In [ ]:
# ANSWER: A stage is a set of tasks that can run without a shuffle.
# Stages are separated by shuffle boundaries (wide transformations).
# Each stage = one or more Tasks running in parallel on partitions.
print("Stage = continuous narrow transformations, bounded by shuffles")

**Q13.** What are the 3 deployment modes for Spark?

In [ ]:
# ANSWER:
# client mode  — driver runs on the submitting machine
# cluster mode — driver runs on a cluster node (most common in production)
# local mode   — driver + executor in one JVM, single machine
print("client | cluster | local")

**Q14.** What does `spark.memory.fraction` control?

In [ ]:
# ANSWER: The fraction of (executor.memory - 300MB reserved) allocated to the
# Unified Memory pool (shared between execution and storage).
# Default: 0.6  Remaining 0.4 is User Memory (UDFs, Python objects)
print(spark.conf.get("spark.memory.fraction"))  # 0.6

**Q15.** What is the medallion architecture? What does each layer guarantee?

In [ ]:
# ANSWER:
# Bronze: Raw data, append-only, schema-on-read, no filtering
# Silver: Cleaned, validated, deduplicated, schema enforced
# Gold:   Aggregated, business-ready, optimised for BI queries
print("Bronze=raw | Silver=trusted | Gold=aggregated")

---
## Section C: Optimisation (6 questions)

**Q16.** What are the 3 AQE runtime optimisations?

In [ ]:
# ANSWER:
# 1. Post-shuffle partition coalescing — merges small partitions
# 2. Convert SortMergeJoin to BroadcastHashJoin — if one side turns out small
# 3. Skew join optimisation — splits skewed partitions
print(spark.conf.get("spark.sql.adaptive.enabled"))  # should be true

**Q17.** When does Dynamic Partition Pruning kick in?

In [ ]:
# ANSWER: DPP applies when:
# 1. Fact table is PARTITIONED on the join key
# 2. Dimension side is small enough to broadcast
# 3. Filter exists on the dimension table
# DPP propagates the dimension filter back to prune fact partitions at scan time.
print(spark.conf.get("spark.sql.optimizer.dynamicPartitionPruning.enabled"))

**Q18.** What is the difference between `coalesce` and `repartition`?

In [ ]:
df = spark.range(1000)
print("Original:",     df.rdd.getNumPartitions())
print("repartition:",  df.repartition(8).rdd.getNumPartitions())   # full shuffle, any count
print("coalesce:",     df.coalesce(2).rdd.getNumPartitions())       # no shuffle, reduce only
# ANSWER: repartition = full shuffle, can increase or decrease
# coalesce = no shuffle, can only DECREASE partitions

---
## Section D: Delta Lake (12 questions)

**Q19.** What does VACUUM do? What is the default retention period?

In [ ]:
# ANSWER: VACUUM removes Parquet files that are no longer part of the current
# table version AND older than the retention window.
# Default retention: 7 days (168 hours).
# After VACUUM, time travel to those versions is impossible.
print("VACUUM default: 7 days retention | OPTIMIZE: compaction only (no delete)")

**Q20.** What are the 4 ACID properties and what does each mean for Delta Lake?

In [ ]:
# ANSWER:
# Atomicity   — write either fully succeeds or fails completely (no partial files)
# Consistency — schema is always valid after any transaction
# Isolation   — concurrent reads/writes don't corrupt each other
# Durability  — committed data survives crashes (transaction log on durable storage)
print("A-C-I-D: Atomic, Consistent, Isolated, Durable")

**Q21.** How do you read a Delta table as it was 3 versions ago?

In [ ]:
DELTA_PATH = tempfile.mkdtemp(prefix="q21_")
for i in range(4):
    spark.createDataFrame([(i,)],["v"]).write.format("delta").mode("append").save(DELTA_PATH)

# Time travel by version
spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH).show()

# By timestamp
# spark.read.format("delta").option("timestampAsOf", "2024-01-01").load(DELTA_PATH)

**Q22.** What are the CDF change types?

In [ ]:
# ANSWER: insert, update_preimage, update_postimage, delete
# update_preimage = value BEFORE update
# update_postimage = value AFTER update
print("insert | update_preimage | update_postimage | delete")

**Q23.** What does OPTIMIZE do? Does it delete old files?

In [ ]:
# ANSWER:
# OPTIMIZE compacts many small Parquet files into fewer, larger files.
# It does NOT delete old files — only marks them as removed in the transaction log.
# Old files are actually deleted by VACUUM after the retention window.
print("OPTIMIZE = compact. Does NOT delete. VACUUM = delete.")

**Q24.** What is schema enforcement in Delta Lake? How do you allow new columns?

In [ ]:
# ANSWER:
# Schema enforcement (default): writing a DataFrame with incompatible/extra columns
# raises an AnalysisException — prevents accidental schema corruption.
#
# To allow new columns (schema evolution):
# df.write.format("delta").option("mergeSchema", "true").mode("append").save(path)
print("mergeSchema=true enables schema evolution")

---
## Section E: Streaming (6 questions)

**Q25.** Why is `checkpointLocation` required in Structured Streaming?

In [ ]:
# ANSWER:
# checkpoint stores: offsets (what data was read), aggregation state, commit log
# Without it:
#   - Restart re-processes all data from the beginning
#   - Stateful aggregations lose accumulated state
# Required for exactly-once semantics.
print("checkpoint = fault tolerance + exactly-once")

**Q26.** What is the watermark formula and what does it control?

In [ ]:
# ANSWER:
# watermark = max(event_time_seen) - delay_threshold
# Events with event_time < watermark are DROPPED (considered too late)
# Windows with end_time < watermark are FINALIZED and emitted
# Watermark bounds the state Spark must keep in memory
print("watermark = max(event_time) - delay")

**Q27.** What is `foreachBatch` and when do you use it?

In [ ]:
# ANSWER:
# foreachBatch(func) calls func(batch_df, batch_id) for each micro-batch
# batch_df is a regular batch DataFrame — any batch API works
# Use cases:
#   - Write to multiple sinks
#   - Delta MERGE (upsert) from streaming data
#   - Apply batch-only operations (JDBC upsert)
print("foreachBatch: custom logic per micro-batch")

---
## Section F: Mixed / Tricky Questions

**Q28.** What is wrong with this code? How do you fix it?
```python
df1 = spark.createDataFrame([(1,"a"),(2,"b")],["id","name"])
df2 = spark.createDataFrame([(1,"x"),(2,"y")],["id","name"])
result = df1.join(df2, df1.id == df2.id)
result.select(col("name")).show()  # AnalysisException: ambiguous
```

In [ ]:
# ANSWER: Both DFs have "name" — ambiguous after join.
# Fix 1: Join on string name (Spark deduplicates the join key)
df1 = spark.createDataFrame([(1,"Alice"),(2,"Bob")],["id","name"])
df2 = spark.createDataFrame([(1,"Eng"),(2,"Mkt")],["id","dept"])  # different col names
df1.join(df2, on="id").show()

# Fix 2: Alias DataFrames before join
df1b = spark.createDataFrame([(1,"Alice")],["id","name"])
df2b = spark.createDataFrame([(1,"Eng")],  ["id","name"])
df1b.alias("a").join(df2b.alias("b"), on="id") \
    .select(col("a.name").alias("emp_name"), col("b.name").alias("dept_name")) \
    .show()

**Q29.** What does `coalesce(col_a, col_b)` do?

In [ ]:
# ANSWER: Returns the first non-null value among its arguments
df = spark.createDataFrame([(None, "B"), ("A", None), (None, None)], ["a","b"])
df.withColumn("result", coalesce(col("a"), col("b"), lit("default"))).show()

**Q30.** What is the import path for the Pandas API on Spark?

In [ ]:
# ANSWER:
import pyspark.pandas as ps
print("import pyspark.pandas as ps  (Spark 3.2+, formerly Koalas)")

psdf = ps.DataFrame({"a": [1,2,3], "b": [4,5,6]})
print(type(psdf))

---

## Score Card

After completing all questions:

| Domain | Questions | My score |
|---|---|---|
| DataFrame API | Q1–Q10 | /10 |
| Architecture | Q11–Q15 | /5 |
| Optimisation | Q16–Q18 | /3 |
| Delta Lake | Q19–Q24 | /6 |
| Streaming | Q25–Q27 | /3 |
| Mixed | Q28–Q30 | /3 |
| **Total** | | **/30** |

**Target: 24+/30 (80%) before taking the real exam.**

**Next:** Day 30 — Full Mock Exam (45 minutes, timed)